# Fase 4 — Tiering Relevansi per Bank (A/B/C/D)

**Input :** `data/interim/news_clean.parquet` (artikel bersih, Fase 3)
**Output:** `data/interim/news_tiered.parquet` — satu baris per **(artikel × bank)**

Skor: `s_b = 3*[judul] + 2*[3 kalimat awal] + min(sebutan badan, 3)`

| Tier | Syarat | Weight | Dipakai |
|------|--------|--------|---------|
| **A** | `s_b >= 3` (spesifik bank) | 1.0 | fitur sentimen per bank |
| **B** | grup (Himbara/bank BUMN) atau regulasi (OJK/BI/LPS) | 0.5 | sektor |
| **C** | pasar/makro (IHSG/rupiah/inflasi/bursa) | 0.0 | konteks (BERTopic) |
| **D** | selain itu | — | dibuang |

Anak usaha (Mandiri Sekuritas, BNI Sekuritas, dll) di-mask → tidak dihitung sebagai bank induk.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import yaml

if Path('src').exists():
    ROOT = Path('.').resolve()
elif Path('../src').exists():
    os.chdir('..'); ROOT = Path('.').resolve()
import sys; sys.path.insert(0, str(ROOT))
from src.tiering import build_tier_rules, tier_articles

INTERIM = ROOT / 'data/interim'
clean = pd.read_parquet(INTERIM / 'news_clean.parquet')
aliases = yaml.safe_load(open(ROOT / 'config/aliases.yaml'))
print('artikel bersih masuk:', len(clean))

In [ ]:
rules = build_tier_rules(aliases)
tiered = tier_articles(clean, rules)

out = INTERIM / 'news_tiered.parquet'
tiered.to_parquet(out, index=False)
print('tersimpan:', out.resolve())
print('kolom    :', list(tiered.columns))

In [ ]:
print('===== DISTRIBUSI TIER =====')
print(tiered.tier.value_counts().to_string())

print('\n===== TIER x BANK =====')
print(pd.crosstab(tiered.tier, tiered.bank.fillna('(pasar)')).to_string())

print('\n===== RINGKASAN =====')
print('artikel bersih     :', len(clean))
print('artikel ter-tier   :', tiered.article_id.nunique(), '(sisanya Tier D, dibuang)')
print('baris (art x bank) :', len(tiered))

print('\n===== TIER A per bank (jumlah artikel spesifik) =====')
print(tiered[tiered.tier=='A'].bank.value_counts().to_string())

In [ ]:
# QC manual: tampilkan 15 contoh Tier A untuk dicek relevansinya (target presisi >=90%)
print('===== CONTOH TIER A (cek manual relevansi) =====')
for _, r in tiered[tiered.tier=='A'].sample(min(15, (tiered.tier=='A').sum()), random_state=42).iterrows():
    print(f'  [{r.bank} s={r.relevance_score}] {r.title[:65]}')

## Setelah selesai
Copy-paste output **DISTRIBUSI TIER**, **TIER x BANK**, **RINGKASAN**, dan contoh **TIER A**
ke chat. Dari sampel Tier A kita cek apakah presisinya tinggi (artikel benar-benar tentang bank itu),
lalu lanjut ke Fase 5 (sentence split + entity linking).